In [1]:
## Main libs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import shutil
import string
import tensorflow as tf
from sklearn.metrics import confusion_matrix

## Preprocessing
import random
from nltk.corpus import wordnet
from transformers import BertTokenizer

import spacy
from spacy.lang.en.examples import sentences

import re
import nltk
from nltk.corpus import stopwords

## Model training
from tensorflow.keras import layers
from tensorflow.keras import losses

from tf_keras import Input, Model, layers
from transformers import TFBertModel, AutoConfig

In [2]:
batch_size = 32
seed = 42
max_features = 10000
sequence_length = 250

train_data = pd.read_csv("/content/sample_data/train_subset.csv")
test_data = pd.read_csv("/content/sample_data/test.csv")

train_data["text"] = (train_data["Title"] + " " + train_data["Description"]).astype(str)
test_data["text"] = (test_data["Title"] + " " + test_data["Description"]).astype(str)

train_data = train_data.sample(frac=1, random_state=seed)

train_size = int(0.8 * len(train_data))
val_data = train_data[train_size:]
train_data = train_data[:train_size]

train_data["Class Index"] -= 1
val_data["Class Index"] -= 1
test_data["Class Index"] -= 1


In [3]:
### PRE-PROCESSING


nltk.download('wordnet')

def synonym_replacement(sentence, n=2):
    words = sentence.split()
    new_words = words.copy()
    random_word_list = list(set([word for word in words if wordnet.synsets(word)]))
    random.shuffle(random_word_list)
    num_replaced = 0
    for random_word in random_word_list:
        synonyms = wordnet.synsets(random_word)
        if not synonyms:
            continue
        synonym_words = set()
        for syn in synonyms:
            for lemma in syn.lemmas():
                synonym_words.add(lemma.name())
        synonym_words.discard(random_word)
        if len(synonym_words) >= 1:
            synonym = random.choice(list(synonym_words))
            new_words = [synonym if word == random_word else word for word in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break
    return " ".join(new_words)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def bert_tokenize(texts, max_length=128):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='tf'
    )

nlp = spacy.load("en_core_web_sm")

def lemmatize_text(text):
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc])


nltk.download('stopwords')
stop_words = set(stopwords.words('english'))


def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [word for word in tokens if word not in stop_words]
    return " ".join(tokens)


def custom_standardization(input_data):
  lowercase = tf.strings.lower(input_data)
  stripped_html = tf.strings.regex_replace(lowercase, '', ' ')
  return tf.strings.regex_replace(stripped_html,
                                  '[%s]' % re.escape(string.punctuation),
                                  '')

def preprocess_text(text):
  text = clean_text(text)
  text = lemmatize_text(text)
  text = synonym_replacement(text, n=2)
  return text

def tokenize_for_bert(texts, labels, max_length=128):
    tokenized = bert_tokenize(texts, max_length=max_length)
    return tokenized, labels

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
train_data['processed_text'] = train_data['text'].apply(preprocess_text)
val_data['processed_text'] = val_data['text'].apply(preprocess_text)
test_data['processed_text'] = test_data['text'].apply(preprocess_text)

# TRAIN DATA
train_texts = train_data['processed_text'].tolist()
train_labels = train_data['Class Index'].values
train_inputs, train_labels = tokenize_for_bert(train_texts, train_labels)

train_inputs_np = {k: np.array(v) for k, v in train_inputs.items()}
train_labels_np = np.array(train_labels)
raw_train_ds = tf.data.Dataset.from_tensor_slices((train_inputs_np, train_labels_np))

# VALIDATION DATA
val_texts = val_data['processed_text'].tolist()
val_labels = val_data['Class Index'].values
val_inputs, val_labels = tokenize_for_bert(val_texts, val_labels)

val_inputs_np = {k: np.array(v) for k, v in val_inputs.items()}
val_labels_np = np.array(val_labels)
raw_val_ds = tf.data.Dataset.from_tensor_slices((val_inputs_np, val_labels_np))

# TEST DATA
test_texts = test_data['processed_text'].tolist()
test_labels = test_data['Class Index'].values
test_inputs, test_labels = tokenize_for_bert(test_texts, test_labels)

test_inputs_np = {k: np.array(v) for k, v in test_inputs.items()}
test_labels_np = np.array(test_labels)
raw_test_ds = tf.data.Dataset.from_tensor_slices((test_inputs_np, test_labels_np))

# BATCH SIZE
train_ds = raw_train_ds.batch(batch_size)
val_ds = raw_val_ds.batch(batch_size)
test_ds = raw_test_ds.batch(batch_size)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


In [5]:
config = AutoConfig.from_pretrained('bert-base-uncased')
bert_encoder = TFBertModel(config)

input_ids = Input(shape=(128,), dtype=tf.int32, name='input_ids')
attention_mask = Input(shape=(128,), dtype=tf.int32, name='attention_mask')
token_type_ids = Input(shape=(128,), dtype=tf.int32, name='token_type_ids')

bert_outputs = bert_encoder(
    input_ids=input_ids,
    attention_mask=attention_mask,
    token_type_ids=token_type_ids
)[1]

x = layers.Dense(16, activation='relu')(bert_outputs)
output = layers.Dense(4, activation='softmax')(x)

model = Model(inputs=[input_ids, attention_mask, token_type_ids], outputs=output)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_ids (InputLayer)      [(None, 128)]                0         []                            
                                                                                                  
 attention_mask (InputLayer  [(None, 128)]                0         []                            
 )                                                                                                
                                                                                                  
 token_type_ids (InputLayer  [(None, 128)]                0         []                            
 )                                                                                                
                                                                                              

In [6]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
epochs = 2
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs)

Epoch 1/2
  5/300 [..............................] - ETA: 4:16:06 - loss: 3.4092 - accuracy: 0.2688

In [ ]:
loss, accuracy = model.evaluate(test_ds)

print("Loss: ", loss)
print("Accuracy: ", accuracy)

In [ ]:
history_dict = history.history
history_dict.keys()

In [ ]:
acc = history_dict['accuracy']
val_acc = history_dict['val_accuracy']
loss = history_dict['loss']
val_loss = history_dict['val_loss']

epochs = range(1, len(acc) + 1)

# "bo" is for "blue dot"
plt.plot(epochs, loss, 'bo', label='Training loss')
# b is for "solid blue line"
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:

plt.plot(epochs, acc, 'bo', label='Training acc')
plt.plot(epochs, val_acc, 'b', label='Validation acc')
plt.title('Training and validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')

plt.show()